# RadarCrédito: Painel de Inadimplência e Risco de Crédito no Brasil (2016-2026)
## Notebook de Exploração Inicial dos Dados (Etapa 2 - Issue #3)

**Disciplina:** Programação Orientada a Objetos  
**Curso:** Ciência de Dados e Inteligência Artificial — PUC-SP  
**Autores:** Grupo do RadarCrédito  

---

### Objetivo do Notebook
Este notebook realiza a **análise exploratória inicial (Etapa 2)** das séries temporais de taxa de inadimplência da carteira de crédito do Sistema Financeiro Nacional, obtidas junto à API do Sistema Gerenciador de Séries Temporais (SGS) do Banco Central do Brasil (BCB).

O propósito é validar a qualidade dos dados, inspecionar tipos de variáveis, verificar ausências ou anomalias, consolidar estatísticas descritivas e estabelecer a base empírica para responder às cinco perguntas centrais do projeto.

---

### Perguntas Norteadoras do Projeto
1. **Evolução e Ciclos:** Como a inadimplência evoluiu nos últimos 10 anos e quais foram os principais picos e vales?
2. **Modalidades Críticas:** Qual modalidade de crédito para pessoa física é a mais inadimplente e a mais volátil?
3. **Sensibilidade à Taxa Básica:** A elevação da taxa Selic antecede aumentos de inadimplência? Com qual defasagem média?
4. **Fatores Macroeconômicos:** A inflação (IPCA) e o nível de endividamento explicam os aumentos da inadimplência?
5. **Ciclo PF vs. PJ:** Os ciclos de inadimplência de pessoa física e de pessoa jurídica seguem o mesmo padrão e sincronia?

---

### Séries Temporais Selecionadas
Todas as séries são de frequência **mensal** e representam a **taxa de inadimplência da carteira de crédito** (percentual da carteira com pelo menos uma parcela em atraso superior a 90 dias):

| Código SGS | Nome da Coluna | Descrição Oficial (BCB) | Segmentação |
|---|---|---|---|
| **21082** | `total` | Inadimplência da carteira de crédito - Total | Carteira Consolidada |
| **21083** | `pessoa_juridica` | Inadimplência da carteira de crédito - Pessoas jurídicas | Segmento PJ |
| **21084** | `pessoa_fisica` | Inadimplência da carteira de crédito - Pessoas físicas | Segmento PF Total |
| **21112** | `pf_livres` | Inadimplência da carteira de crédito - PF - Recursos livres | PF Taxas Livres |
| **21113** | `pf_cheque_especial` | Inadimplência da carteira de crédito - PF - Cheque especial | PF Crédito Rotativo |
| **21129** | `pf_cartao` | Inadimplência da carteira de crédito - PF - Cartão de crédito total | PF Crédito Rotativo/Parcelado |
| **21145** | `pf_direcionados` | Inadimplência da carteira de crédito - PF - Recursos direcionados | PF Crédito Habitacional/Rural |

**Estrutura de Hierarquia:**
- `total` = ponderação entre `pessoa_fisica` e `pessoa_juridica`.
- `pessoa_fisica` = ponderação entre recursos livres (`pf_livres`) e direcionados (`pf_direcionados`).
- Recursos livres (`pf_livres`) contemplam linhas de alto custo e rotativo (`pf_cheque_especial`, `pf_cartao`), além de consignado e CDC.

In [1]:
import sys
from pathlib import Path

# Ajuste do caminho raiz do projeto para funcionamento em notebooks/ ou na raiz
raiz = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Configura renderer padrão do Plotly para compatibilidade com notebooks
pio.renderers.default = "notebook_connected+plotly_mimetype"

from src.fonte_dados import FonteBCB
from src.analisador import AnalisadorDados

# Instancia a fonte com pasta de cache apontando para a pasta data do projeto
fonte = FonteBCB(pasta_cache=str(raiz / "data"))
print(f"Fonte configurada: {fonte.nome}")
print(f"Pasta de cache: {fonte.pasta_cache}")

Fonte configurada: Banco Central do Brasil (SGS)
Pasta de cache: C:\Users\paulo\Desktop\Projetos\Rad-cred\radar-credito\data


## Seção 1. Carregamento das Séries Temporais

Nesta seção, consultamos as 7 séries de crédito selecionadas para o recorte decenal solicitado: **01/11/2016 a 31/10/2026**.
Utilizamos a classe `FonteBCB`, que abstrai as requisições HTTP à API do SGS do Banco Central e implementa política de contingência com cache local em arquivo CSV (`data/serie_{codigo}.csv`).

In [2]:
# Mapeamento oficial de colunas e códigos SGS do BCB
series_codigos = {
    "total": 21082,
    "pessoa_juridica": 21083,
    "pessoa_fisica": 21084,
    "pf_livres": 21112,
    "pf_cheque_especial": 21113,
    "pf_cartao": 21129,
    "pf_direcionados": 21145,
}

data_inicio = "01/11/2016"
data_fim = "31/10/2026"

series = {}
print("Carregando séries temporais do Banco Central do Brasil:")
print("-" * 75)

for nome, codigo in series_codigos.items():
    tabela = fonte.obter_serie(codigo, data_inicio, data_fim)
    series[nome] = tabela
    data_min = tabela["data"].min().strftime("%d/%m/%Y")
    data_max = tabela["data"].max().strftime("%d/%m/%Y")
    qtd_linhas = len(tabela)
    print(f"Série {codigo:5d} ({nome:19s}): {qtd_linhas} registros | {data_min} a {data_max}")

print("-" * 75)
print("Amostra da série 'total' (cinco primeiras linhas):")
display(series["total"].head())

print("Amostra da série 'total' (cinco últimas linhas):")
display(series["total"].tail())

Carregando séries temporais do Banco Central do Brasil:
---------------------------------------------------------------------------
Série 21082 (total              ): 119 registros | 01/11/2016 a 01/09/2026
Série 21083 (pessoa_juridica    ): 119 registros | 01/11/2016 a 01/09/2026
Série 21084 (pessoa_fisica      ): 119 registros | 01/11/2016 a 01/09/2026
Série 21112 (pf_livres          ): 119 registros | 01/11/2016 a 01/09/2026
Série 21113 (pf_cheque_especial ): 119 registros | 01/11/2016 a 01/09/2026
Série 21129 (pf_cartao          ): 119 registros | 01/11/2016 a 01/09/2026
Série 21145 (pf_direcionados    ): 119 registros | 01/11/2016 a 01/09/2026
---------------------------------------------------------------------------
Amostra da série 'total' (cinco primeiras linhas):
Amostra da série 'total' (cinco últimas linhas):


,data,valor
0,2016-11-01,3.86
1,2016-12-01,3.75
2,2017-01-01,3.79
3,2017-02-01,3.82
4,2017-03-01,3.91


,data,valor
114,2026-05-01,4.75
115,2026-06-01,4.59
116,2026-07-01,4.88
117,2026-08-01,5.05
118,2026-09-01,4.98


### Interpretação do Carregamento
- Cada série retornou **119 registros mensais**, compreendidos entre **01/11/2016 e 01/09/2026**.
- A defasagem entre a data final solicitada (31/10/2026) e o último dado disponível (setembro/2026) decorre da política de publicação do Banco Central (defasagem operacional de cerca de 45 a 60 dias para apuração do risco de crédito consolidado).
- O mecanismo de contingência em `data/` assegurou a integridade dos dados mesmo sob indisponibilidade transitória da rede.

## Seção 2. Identificação de Variáveis e Tipos de Dados

O Banco Central disponibiliza dados via JSON com campos textuais (`"data"` como string no formato `"dd/mm/aaaa"` e `"valor"` como string numérica decimal com separador de ponto).
Nesta seção, confirmamos os tipos e mapeamos cada variável do domínio do problema.

In [3]:
print("--- Informações estruturais da série 'total' ---")
series["total"].info()

print("\nTipos de dados no DataFrame:")
display(series["total"].dtypes)

# Tabela resumo metadados das variáveis
metadados = [
    {
        "variavel": "data",
        "tipo_pandas": "datetime64[ns]",
        "papel": "Dimensão Temporal",
        "unidade": "Data (AAAA-MM-DD)",
        "periodicidade": "Mensal (início de mês)",
        "fonte": "Banco Central do Brasil (SGS)",
        "descricao": "Mês de referência da apuração da carteira"
    },
    {
        "variavel": "valor",
        "tipo_pandas": "float64",
        "papel": "Métrica Quantitativa",
        "unidade": "% da carteira",
        "periodicidade": "Mensal",
        "fonte": "Banco Central do Brasil (SGS)",
        "descricao": "Taxa de inadimplência (atraso > 90 dias)"
    }
]

tabela_metadados = pd.DataFrame(metadados)
print("\nTabela Resumo das Variáveis:")
display(tabela_metadados)

--- Informações estruturais da série 'total' ---
<class 'pandas.DataFrame'>
RangeIndex: 119 entries, 0 to 118
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    119 non-null    datetime64[us]
 1   valor   119 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 2.0 KB

Tipos de dados no DataFrame:

Tabela Resumo das Variáveis:


data     datetime64[us]
valor           float64
dtype: object

,variavel,tipo_pandas,papel,unidade,periodicidade,fonte,descricao
0,data,datetime64[ns],Dimensão Temporal,Data (AAAA-MM-DD),Mensal (início de mês),Banco Central do Brasil (SGS),Mês de referência da apuração da carteira
1,valor,float64,Métrica Quantitativa,% da carteira,Mensal,Banco Central do Brasil (SGS),Taxa de inadimplência (atraso > 90 dias)


### Interpretação dos Tipos e Conversão
- A coluna `data` foi convertida de string para `datetime64[ns]` via `pd.to_datetime(tabela["data"], format="%d/%m/%Y")` em `FonteBCB`. Isso permite operações temporais precisas (filtragem, ordenação cronológica e agrupamento por ano).
- A coluna `valor` foi convertida com `pd.to_numeric(tabela["valor"], errors="coerce")`, que transforma eventuais strings defeituosas ou vazias em `NaN`, viabilizando cálculos estatísticos float64 sem quebras de execução.